# A character-level RNN that writes Shakespeare

Notebook 5 turned text into integers and explained how `nn.Embedding` learns a dense vector for each token. Now we put those vectors to work in an actual **sequence model**.

Up to now our models have been **stateless**: a fixed-size input goes in, a fixed-size output comes out. That's fine for digits or for fitting curves, but it's a bad fit for sequences — sentences, audio, time series — where the meaning of a token depends on *what came before*.

A **recurrent neural network (RNN)** fixes this by carrying a **hidden state** $\mathbf{h}_t$ from one step to the next. At each timestep, the network reads an input $\mathbf{x}_t$ *and* the previous hidden state $\mathbf{h}_{t-1}$, and produces a new hidden state plus an output.

The simplest RNN — the "Elman cell" — is just two linear layers and a non-linearity:

$$
\mathbf{h}_t = \tanh(W_{xh}\,\mathbf{x}_t + W_{hh}\,\mathbf{h}_{t-1} + \mathbf{b}_h)
$$

$$
\mathbf{y}_t = W_{hy}\,\mathbf{h}_t + \mathbf{b}_y
$$

That's it. The same weights $(W_{xh}, W_{hh}, W_{hy})$ are used at every timestep — the recurrence is just feeding $\mathbf{h}_t$ back in at step $t+1$.

### Unrolled in time

```
   x₁           x₂           x₃           x₄          ...
   │            │            │            │
   ▼            ▼            ▼            ▼
 ┌────┐  W_hh ┌────┐  W_hh ┌────┐  W_hh ┌────┐
 │ h₁ │ ───► │ h₂ │ ───► │ h₃ │ ───► │ h₄ │  ...
 └────┘       └────┘       └────┘       └────┘
   │            │            │            │
   ▼            ▼            ▼            ▼
   y₁           y₂           y₃           y₄          ...
```

Everything we've already learned — autograd, optimizers, loss functions — applies unchanged. The only new wrinkle is that we run a **for-loop** over the sequence inside the model's `forward`.

In this notebook we'll:

1. Reload the tokenized Shakespeare data from notebook 5.
2. Build an Elman RNN *from scratch* with two `nn.Linear` layers.
3. Train it to predict the next character given the previous ones.
4. **Generate fresh Shakespeare-flavored text** by sampling autoregressively, with controllable temperature.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import datetime as dt
import urllib.request

import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
import torch
import torch.nn as nn
from torch.utils.tensorboard import SummaryWriter
from tqdm.notebook import tqdm

pio.renderers.default = "notebook"

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(1337)
print(f"PyTorch: {torch.__version__}")
print(f"Device:  {device}")

## Reload the tokenized corpus

This is the same loading + tokenization recipe from notebook 5, compressed into one cell so the RNN notebook is self-contained. If you already ran notebook 5, the `input.txt` file is already on disk; otherwise this downloads it.

In [ ]:
data_path = "data/shakespeare_char/input.txt"
if not os.path.exists(data_path):
    os.makedirs(os.path.dirname(data_path), exist_ok=True)
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    print(f"Downloading {url} ...")
    urllib.request.urlretrieve(url, data_path)

with open(data_path, "r") as f:
    text = f.read()

# Char-level vocab
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s):  return [stoi[c] for c in s]
def decode(xs): return "".join(itos[int(i)] for i in xs)

# Tokenize the whole corpus + train/val split
data = torch.tensor(encode(text), dtype=torch.long)
n_train = int(0.9 * len(data))
train_data, val_data = data[:n_train], data[n_train:]

print(f"Corpus:     {len(text):,} chars")
print(f"Vocab size: {vocab_size}")
print(f"Train:      {len(train_data):,} tokens")
print(f"Val:        {len(val_data):,} tokens")

## The model

We build the Elman cell by hand using `nn.Linear`s — no `nn.RNN` shortcut, so the recurrence is visible in `forward`.

A few small additions on top of the bare equations:

- An **embedding table** turns each integer character into a learned dense vector (much friendlier than one-hot inputs).
- We process a **batch of sequences** at once for GPU efficiency. The hidden state has shape `(B, H)`; the per-step input has shape `(B, H)` after embedding; the for-loop runs over the time dimension.
- The output projection $W_{hy}$ produces **logits** over the full vocabulary at every timestep.

The training signal is straightforward: given a chunk of characters $x_0, x_1, \ldots, x_{T-1}$, predict the *next* characters $x_1, x_2, \ldots, x_T$ at every position. That's one cross-entropy loss per timestep, averaged.

In [ ]:
class CharRNN(nn.Module):
    """Bare Elman RNN: h_t = tanh(W_xh x_t + W_hh h_{t-1} + b)."""

    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.embed = nn.Embedding(vocab_size, hidden_size)
        # The recurrence: two linear layers (we fold b_h into W_hh's bias)
        self.W_xh = nn.Linear(hidden_size, hidden_size, bias=False)
        self.W_hh = nn.Linear(hidden_size, hidden_size, bias=True)
        # Output projection: hidden -> vocab logits
        self.W_hy = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, h=None):
        """
        x: (B, T) integer token ids
        h: (B, H) initial hidden state, or None (zeros)
        returns: logits (B, T, V), final hidden (B, H)
        """
        B, T = x.shape
        if h is None:
            h = torch.zeros(B, self.hidden_size, device=x.device)
        emb = self.embed(x)  # (B, T, H)

        logits = []
        for t in range(T):
            h = torch.tanh(self.W_xh(emb[:, t]) + self.W_hh(h))
            logits.append(self.W_hy(h))
        return torch.stack(logits, dim=1), h  # (B, T, V), (B, H)


HIDDEN = 256
model = CharRNN(vocab_size, HIDDEN).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"\nTrainable parameters: {n_params:,}")

## Training

We don't pre-tile the data into batches; instead we sample `batch_size` random starting positions from the training tensor on each step, slice `block_size + 1` tokens, and split into `(x, y)` where `y` is `x` shifted by one. That's the textbook recipe for character language modeling and avoids any DataLoader plumbing.

The training loss is the average cross-entropy of predicted-vs-actual *next* character across all positions. **Gradient clipping** (`clip_grad_norm_`) is important for vanilla RNNs — without it, occasional exploding gradients will blow up the parameters.

Open `./scripts/tensorboard.sh` in another terminal to watch the loss curves live. Every iteration also logs a sample generation so you can watch the model's babble cohere over time in TensorBoard's "Text" tab.

In [ ]:
@torch.no_grad()
def generate(model, prompt="\n", max_new_tokens=400, temperature=1.0):
    """Autoregressively sample from the model. temperature>1: more random; <1: greedier."""
    model.eval()
    ids = torch.tensor(encode(prompt), dtype=torch.long, device=device).unsqueeze(0)
    # Warm up the hidden state with the prompt
    _, h = model(ids)
    out = list(ids[0].tolist())
    last = ids[:, -1:]
    for _ in range(max_new_tokens):
        logits, h = model(last, h)
        logits = logits[:, -1, :] / max(temperature, 1e-5)
        probs = torch.softmax(logits, dim=-1)
        nxt = torch.multinomial(probs, num_samples=1)
        out.append(int(nxt))
        last = nxt
    model.train()
    return decode(out)


def get_batch(split, block_size, batch_size):
    data_ = train_data if split == "train" else val_data
    ix = torch.randint(0, len(data_) - block_size - 1, (batch_size,))
    x = torch.stack([data_[i:i + block_size]     for i in ix])
    y = torch.stack([data_[i + 1:i + 1 + block_size] for i in ix])
    return x.to(device), y.to(device)


# ---- hyperparameters ----
block_size = 128
batch_size = 64
n_iters    = 3000
lr         = 3e-3
eval_every = 250
grad_clip  = 5.0

# ---- optimizer + logger ----
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
run_name  = f"rnn-h{HIDDEN}-b{block_size}-{dt.datetime.now().strftime('%Y%m%d-%H%M%S')}"
log_dir   = f"runs/{run_name}"
writer    = SummaryWriter(log_dir)
print(f"Logging to: {log_dir}\n")


@torch.no_grad()
def estimate_val_loss(n_batches=20):
    model.eval()
    losses = []
    for _ in range(n_batches):
        xv, yv = get_batch("val", block_size, batch_size)
        lg, _ = model(xv)
        losses.append(nn.functional.cross_entropy(lg.reshape(-1, vocab_size), yv.reshape(-1)).item())
    model.train()
    return float(np.mean(losses))


# ---- training loop ----
model.train()
for step in tqdm(range(n_iters), desc="training"):
    x, y = get_batch("train", block_size, batch_size)
    logits, _ = model(x)
    loss = nn.functional.cross_entropy(logits.reshape(-1, vocab_size), y.reshape(-1))

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    optimizer.step()

    writer.add_scalar("train/loss", loss.item(), step)
    if step % eval_every == 0 or step == n_iters - 1:
        val = estimate_val_loss()
        writer.add_scalar("val/loss", val, step)
        sample = generate(model, prompt="\n", max_new_tokens=200, temperature=0.9)
        writer.add_text("samples", f"```\n{sample}\n```", step)
        print(f"step {step:5d}  train_loss={loss.item():.4f}  val_loss={val:.4f}")

writer.close()
print("\nDone.")

## Sample some Shakespeare

Now the fun part. Given the trained weights, we generate text by feeding the model a starting prompt, taking its predicted distribution over the next character, sampling one, and feeding *that* back as the next input. Repeat until we've made enough characters.

The **temperature** knob controls how peaky the sampling distribution is:

- $\tau \to 0$ — argmax, fully deterministic. Often gets stuck in repetitive loops.
- $\tau = 1$ — sample directly from the model's predictions.
- $\tau > 1$ — flatter distribution, more wild / nonsensical.

Try them side by side.

In [ ]:
for temp in (0.3, 0.7, 1.0, 1.3):
    print(f"\n{'=' * 70}")
    print(f"  temperature = {temp}")
    print('=' * 70)
    print(generate(model, prompt="\nROMEO:\n", max_new_tokens=500, temperature=temp))

## What you should see, and what's still wrong

After ~3000 iterations a vanilla Elman RNN typically reaches a val loss of around **1.5–1.7 nats/char**. The output looks *Shakespeare-y* at a glance:

- Sentences end in punctuation and start with capitals.
- Character names sit on their own line followed by a colon (`ROMEO:`).
- Words mostly *look* like English words.

But pay attention and the cracks show:

- Most generated "words" aren't real words.
- The model can't keep track of a coherent thought beyond ~20 characters. By the time it's written a few words, it has forgotten where the sentence was going.

This is the famous **vanishing gradient** problem of vanilla RNNs: information from far back in the sequence has to pass through `tanh(W_hh · …)` many times to influence the present, and that signal gets crushed exponentially fast. **LSTMs** and **GRUs** were invented specifically to fix this (with explicit "remember / forget" gates), and would be a drop-in upgrade — `nn.LSTM` would replace the hand-rolled cell, and you'd see noticeably better generations at the same model size.

But the real punchline is that the architecture we use in `model.py` of this repo — the **Transformer** — sidesteps the recurrence entirely. Every position attends directly to every other position in the sequence in parallel, so there's no long chain of `tanh`s to send gradients through. That's why modern language models are transformers, not RNNs.

This RNN is the small bridge between "fitting a curve with PyTorch" and "what `train.py` in nanoGPT actually does". Same training loop, same loss function, just a different `forward`.